In [1]:
import numpy as np
import pandas as pd

PIP = 0.0001
COST_PIPS = 1.4          # spread + slippage, round trip (from the specs)

def simulate_trade(bars, entry_i, side, stop, target, max_bars=None, exit_time=None):
    """
    bars:      DataFrame with open/high/low/close, index = bar start time (UTC)
    entry_i:   integer position of the bar we ENTER on (fill at its open)
    side:      +1 long, -1 short
    stop, target: absolute prices
    max_bars:  close at that bar's close if still open after this many bars
    exit_time: 'HH:MM' UTC, close at the open of the first bar at/after this time
    Returns a dict describing the trade, with P&L in pips after costs.
    """
    entry_px = bars["open"].iat[entry_i]
    n = len(bars)
    for k, i in enumerate(range(entry_i, n)):
        t = bars.index[i]
        o, h, l, c = bars["open"].iat[i], bars["high"].iat[i], bars["low"].iat[i], bars["close"].iat[i]

        # time exit: leave at this bar's open
        if exit_time is not None and k > 0 and t.strftime("%H:%M") >= exit_time:
            exit_px, reason = o, "time"
            break

        if side == 1:
            hit_stop, hit_tgt = l <= stop, h >= target
        else:
            hit_stop, hit_tgt = h >= stop, l <= target

        if hit_stop:                       # checked FIRST: worst case if both touched
            # if the bar opened beyond the stop (a gap), we fill at the open, not the stop
            gap = (o < stop) if side == 1 else (o > stop)
            exit_px, reason = (o if gap else stop), "stop"
            break
        if hit_tgt:
            exit_px, reason = target, "target"
            break
        if max_bars is not None and k + 1 >= max_bars:
            exit_px, reason = c, "max_bars"
            break
    else:
        exit_px, reason, i = bars["close"].iat[n - 1], "end_of_data", n - 1

    gross = side * (exit_px - entry_px) / PIP
    return {
        "entry_time": bars.index[entry_i], "exit_time": bars.index[i],
        "side": side, "entry": entry_px, "exit": exit_px, "reason": reason,
        "bars_held": i - entry_i + 1,
        "pips_gross": gross, "pips_net": gross - COST_PIPS,
        "risk_pips": abs(entry_px - stop) / PIP,
    }

In [2]:
import numpy as np
import pandas as pd
from sqlalchemy import create_engine

engine = create_engine("postgresql+psycopg2://postgres@127.0.0.1:5432/trading")

def load_bars(tf):
    b = pd.read_sql(
        "SELECT ts_utc, open, high, low, close, atr_14, session FROM fx_bars "
        "WHERE pair = 'EURUSD' AND timeframe = %(tf)s ORDER BY ts_utc",
        engine, params={"tf": tf}, index_col="ts_utc",
    )
    b.index = pd.to_datetime(b.index, utc=True)
    return b

b15 = load_bars("15m")
b60 = load_bars("1h")
b15 = b15[b15.index < "2022-01-01"]     # BUILD PERIOD ONLY
b60 = b60[b60.index < "2022-01-01"]
print(len(b15), len(b60))

174071 43527


In [3]:
def run_strategy_a(b15, b60, stop_mult=1.0, tgt_mult=2.0):
    trades, skipped = [], {"no_data": 0, "range_too_wide": 0, "range_too_narrow": 0, "no_breakout": 0}
    pos = {t: i for i, t in enumerate(b15.index)}          # timestamp -> row number

    asia_rng = (b15.between_time("00:00", "06:45")
                  .groupby(b15.between_time("00:00", "06:45").index.normalize())
                  .agg(hi=("high", "max"), lo=("low", "min")))
    asia_rng["rng"] = asia_rng["hi"] - asia_rng["lo"]
    asia_rng["cap"] = asia_rng["rng"].shift(1).rolling(60, min_periods=40).quantile(0.80)

    for day, g in b15.groupby(b15.index.normalize()):
        asia = g.between_time("00:00", "06:45")
        london = g.between_time("07:00", "10:45")
        atr_bar = day + pd.Timedelta(hours=6)              # 06:00 1h bar, finished at 07:00
        if len(asia) < 20 or london.empty or atr_bar not in b60.index:
            skipped["no_data"] += 1; continue

        hi, lo = asia["high"].max(), asia["low"].min()
        atr = b60.at[atr_bar, "atr_14"]
        rng = hi - lo
        cap = asia_rng["cap"].get(day, np.nan)
        if np.isnan(cap) or np.isnan(atr):  skipped["no_data"] += 1;        continue
        if rng > cap:                       skipped["range_too_wide"] += 1; continue
        if rng < 10 * PIP:       skipped["range_too_narrow"] += 1; continue

        signal = None
        for t, row in london.iterrows():
            if row["close"] > hi: signal = (t, 1);  break
            if row["close"] < lo: signal = (t, -1); break
        if signal is None:
            skipped["no_breakout"] += 1; continue

        t, side = signal
        entry_i = pos[t] + 1                                # fill at the NEXT bar's open
        if entry_i >= len(b15) or b15.index[entry_i].normalize() != day:
            skipped["no_data"] += 1; continue
        entry_px = b15["open"].iat[entry_i]
        stop   = entry_px - side * stop_mult * atr
        target = entry_px + side * tgt_mult * atr
        tr = simulate_trade(b15, entry_i, side, stop, target, exit_time="16:00")
        tr["atr_pips"] = atr / PIP
        trades.append(tr)

    return pd.DataFrame(trades), skipped

tradesA, skippedA = run_strategy_a(b15, b60)
print(skippedA)
print(len(tradesA), "trades")
tradesA["reason"].value_counts()

{'no_data': 411, 'range_too_wide': 364, 'range_too_narrow': 68, 'no_breakout': 75}
1267 trades


reason
stop      850
target    395
time       22
Name: count, dtype: int64

In [4]:
t = tradesA
t["R"] = t["pips_net"] / t["risk_pips"]          # result in units of risk (1R = lost the stop)
wins = t[t.pips_net > 0]; losses = t[t.pips_net <= 0]
print(f"trades: {len(t)}   win rate: {len(wins)/len(t):.1%}")
print(f"avg win: {wins.pips_net.mean():.1f} pips   avg loss: {losses.pips_net.mean():.1f} pips")
print(f"net pips total: {t.pips_net.sum():.0f}   (gross before costs: {t.pips_gross.sum():.0f})")
print(f"profit factor: {wins.pips_net.sum() / -losses.pips_net.sum():.2f}")
print(f"avg R per trade: {t.R.mean():.3f}")

trades: 1267   win rate: 32.2%
avg win: 18.8 pips   avg loss: -11.3 pips
net pips total: -2064   (gross before costs: -291)
profit factor: 0.79
avg R per trade: -0.199


In [5]:
print(skippedA)
print(tradesA["atr_pips"].describe())
print(tradesA.groupby("side")["R"].agg(["count", "mean"]))

{'no_data': 411, 'range_too_wide': 364, 'range_too_narrow': 68, 'no_breakout': 75}
count    1267.000000
mean       10.098608
std         3.622047
min         4.007143
25%         7.560714
50%         9.485714
75%        11.782143
max        28.450000
Name: atr_pips, dtype: float64
      count      mean
side                 
-1      632 -0.169918
 1      635 -0.228057


In [6]:
t = tradesA.copy()
t["R"] = t["pips_net"] / t["risk_pips"]
t["R_gross"] = t["pips_gross"] / t["risk_pips"]
print(t.groupby(t.entry_time.dt.year)[["R", "R_gross"]].agg(["count", "mean"]).round(3))
print(t["reason"].value_counts(normalize=True).round(3))

               R        R_gross       
           count   mean   count   mean
entry_time                            
2015         171 -0.113     171 -0.010
2016         184 -0.185     184 -0.047
2017         195 -0.226     195 -0.078
2018         181 -0.202     181 -0.061
2019         160 -0.416     160 -0.204
2020         176 -0.168     176 -0.013
2021         200 -0.109     200  0.080
reason
stop      0.671
target    0.312
time      0.017
Name: proportion, dtype: float64


In [7]:
grid = []
for s in [0.75, 1.0, 1.5]:
    for g in [1.5, 2.0, 3.0]:
        tr, _ = run_strategy_a(b15, b60, stop_mult=s, tgt_mult=g)
        R = tr["pips_net"] / tr["risk_pips"]
        grid.append({"stop": s, "target": g, "trades": len(tr),
                     "win_rate": (tr.pips_net > 0).mean(),
                     "avg_R": R.mean(), "total_R": R.sum()})
grid = pd.DataFrame(grid).round(3)
print(grid.pivot(index="stop", columns="target", values="avg_R"))
grid.sort_values("avg_R", ascending=False)

target    1.5    2.0    3.0
stop                       
0.75   -0.221 -0.244 -0.217
1.00   -0.181 -0.199 -0.178
1.50   -0.146 -0.154 -0.142


,stop,target,trades,win_rate,avg_R,total_R
8,1.50,3.0,1267,0.337,-0.142,-180.022
6,1.50,1.5,1267,0.476,-0.146,-184.553
7,1.50,2.0,1267,0.406,-0.154,-195.737
5,1.00,3.0,1267,0.258,-0.178,-225.976
3,1.00,1.5,1267,0.390,-0.181,-229.145
4,1.00,2.0,1267,0.322,-0.199,-252.205
2,0.75,3.0,1267,0.208,-0.217,-275.335
0,0.75,1.5,1267,0.329,-0.221,-279.988
1,0.75,2.0,1267,0.265,-0.244,-308.790
